In [ ]:
%connect "<project_name>"

#### Create Foreign table to access your native object store data

In [ ]:
-- Create Authorization
CREATE AUTHORIZATION "<project_user_name>".Auth_S3
AS INVOKER TRUSTED
USER '<environment_user_name>'
PASSWORD '<environment_user_password>';

In [ ]:
CREATE multiset FOREIGN TABLE WEATHER,
EXTERNAL SECURITY INVOKER TRUSTED Auth_S3
(
      ID BIGINT,
      "Date" VARCHAR(1024),
      Loc VARCHAR(1024),    
      MinTemp FLOAT,
      MaxTemp FLOAT,
      Rainfall FLOAT,
      Evaporation FLOAT,
      Sunshine FLOAT,
      WindGustDir VARCHAR(1024),
      WindGustSpeed FLOAT,
      WindDir9am VARCHAR(1024),
      WindDir3pm VARCHAR(1024),
      WindSpeed9am FLOAT,
      WindSpeed3pm FLOAT,
      Humidity9am FLOAT,
      Humidity3pm FLOAT,
      Pressure9am FLOAT,
      Pressure3pm FLOAT,
      Cloud9am FLOAT,
      Cloud3pm FLOAT,
      Temp9am FLOAT,
      Temp3pm FLOAT,
      RainToday VARCHAR(1024),
      rainfalltomorrow FLOAT,
      RainTomorrow VARCHAR(1024)
)
USING
(
  LOCATION('/S3/s3.amazonaws.com/rli-data/WEATHER')
  STOREDAS ('PARQUET')
)
NO PRIMARY INDEX
PARTITION BY COLUMN;

<h3 style="padding:0px 0px;">1.1.1 What are the top 3 features with most missing values</h3>

In [ ]:
select ColumnName, NullCount, NullPercentage from TD_ColumnSummary (
  on weather as InputTable
  using
  TargetColumns ('[:]')
) as dt ORDER BY NullCount desc;

In [ ]:
%chart x=ColumnName, y=NullPercentage, title="Features with Missing Values", labelx=Feature, labely=%Null

<h3 style="padding:0px 0px;">1.1.2 How many features have negative values in the dataset</h3>

In [ ]:
select ColumnName, NegativeCount from TD_ColumnSummary (
  on weather as InputTable
  using
  TargetColumns ('[:]')
) as dt ORDER BY NegativeCount desc;

In [ ]:
%chart x=ColumnName, y=NegativeCount, title="Features with Negative Values", labelx=Feature, labely=#Negatives

<h3 style="padding:0px 0px;">1.1.3 What is the count of rainy days versus non-rainy days</h3>

In [ ]:
select * from TD_CategoricalSummary (
  on weather as InputTable
  using
    TargetColumns ('RainToday')
) as dt;

In [ ]:
%chart x=DistinctValue, y=DistinctValueCount, title="Rainy days versus non-Rainy days", labelx=Category, labely=#Count

<h3 style="padding:0px 0px;">1.1.4 How many distinct categories are there for Location</h3>

In [ ]:
select * from TD_CategoricalSummary (
  on weather as InputTable
  using
    TargetColumns ('Loc')
) as dt;

In [ ]:
%chart x=DistinctValue, y=DistinctValueCount, title="Categories for Location", labelx=Loc, labely=#Count

<h3 style="padding:0px 0px;">1.1.5 What is the mean and standard deviation for MaxTemp? Does it follow normal distribution ? </h3>

In [ ]:
select * from TD_UnivariateStatistics (
  on weather as InputTable
  using
  TargetColumns ('MaxTemp')
  Stats('MEAN','STD')
) as dt ORDER BY 1,2;

In [ ]:
select id, w."Date" AS "Day", w.MaxTemp from weather as w
where w.MaxTemp is NOT NULL
ORDER BY id;

In [ ]:
%chart x=id, y=MaxTemp, title="MaxTemp Values", labelx=Day, labely=MaxTemp, mark=point, height=500, width=1400

In [ ]:
select * from TD_QQNorm (
  on (select maxtemp, cast (row_number() over (order by maxtemp asc nulls last) as bigint) as rank_maxtemp from weather) as InputTable
  USING
  TargetColumns('maxtemp')
  RankColumns('rank_maxtemp')
) as dt;

In [ ]:
%chart x=MaxTemp, y=MaxTemp_theoretical_quantiles, title="QQPlot", labelx=Theoretical Quantiles, labely=Sample Quantiles, mark=point

In [ ]:
select * from td_histogram (
  on weather as InputTable
  USING 
  MethodType ('Equal-width')
  TargetColumn ('MaxTemp')
  NBins(30)
) as dt;

In [ ]:
%chart x=Minvalue, y=CountOfValues, title="MaxTemp Histogram", labelx=MaxTemp, labely=# values

In [ ]:
DROP TABLE WEATHER

In [ ]:
DROP AUTHORIZATION "<project_user_name>".Auth_S3